# Seaquest Stage-S1 — Vanilla Action-Conditioned State Critic

**Question:** can a vanilla Eysenbach-style state critic actually learn AND use the action input
on the validated Seaquest observational data? (No oxygen-confounder work; no pixels; no actor;
no causal/robust losses; no data regeneration in Colab.)

This notebook is **self-contained**: it loads the frozen `seaquest_s1_colab_pack.zip` exported
locally from the corrected Stage-S0.5 run, trains the critic (Models A/B/C, seed 0), runs frozen
evaluation (representation, action-shuffle, same-state sensitivity, forced-branch alignment),
applies gates C1-C5, and exports checkpoints/metrics/figures/SUMMARY.

Forbidden at runtime: EnvPool, OCAtari, ALE, ROMs, JAX/Flax, the CleanRL teacher, data regen.


## Section 1 — Setup


In [ ]:
import os, sys, json, io, zipfile, hashlib, time, math, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

# --- optional Google Drive (uncomment to mount; pack can also be uploaded) ---
# from google.colab import drive; drive.mount('/content/drive')

# --- (optional) clone repo at an explicit reviewed commit for reference only ---
# !git clone https://github.com/tingrui-huang/Goal-Conditioned-Confounded-MsPacman /content/repo
# %cd /content/repo && !git checkout <REVIEWED_COMMIT>

SEED = 0
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CONFIG = dict(optimizer="Adam", lr=3e-4, batch_size=256, max_epochs=60,
              patience=8, weight_decay=0.0, grad_clip=10.0, emb_dim=128,
              horizon=16, primary_seed=0, checkpoint_criterion="lowest_val_NCE")
print("Python", sys.version.split()[0], "| PyTorch", torch.__version__,
      "| CUDA", torch.cuda.is_available(), "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("device", DEVICE, "| seed", SEED)
print("config", json.dumps(CONFIG))


## Frozen model / loss / evaluation code (embedded; do not edit)


In [ ]:
# === critic.py ===
"""Stage-S1 vanilla action-conditioned contrastive state critic (PyTorch only).

f(s,a,g) = phi(s,a)^T psi(g) / sqrt(d).  Eysenbach-style. NO pixels, NO RNN/attention/
residual, NO weight sharing between phi and psi. A matched-capacity no-action baseline
f(s,g) uses the SAME encoders minus the action input.
"""
import torch
import torch.nn as nn

EMB_DIM = 128
ACTION_DIM = 18


def _mlp(in_dim, out_dim=EMB_DIM):
    return nn.Sequential(
        nn.Linear(in_dim, 256), nn.ReLU(),
        nn.Linear(256, 256), nn.ReLU(),
        nn.Linear(256, out_dim))


class StateActionCritic(nn.Module):
    """phi(state ⊕ one_hot(action)) and psi(goal); score = phi·psi / sqrt(d)."""
    def __init__(self, state_dim, goal_dim, action_dim=ACTION_DIM, emb_dim=EMB_DIM):
        super().__init__()
        self.state_dim = state_dim; self.goal_dim = goal_dim
        self.action_dim = action_dim; self.emb_dim = emb_dim
        self.use_action = True
        self.phi = _mlp(state_dim + action_dim, emb_dim)
        self.psi = _mlp(goal_dim, emb_dim)
        self._scale = emb_dim ** 0.5

    def encode_sa(self, states, actions_onehot):
        x = torch.cat([states, actions_onehot], dim=-1)
        return self.phi(x)

    def encode_g(self, goals):
        return self.psi(goals)

    def score_matrix(self, sa_repr, g_repr):
        return sa_repr @ g_repr.t() / self._scale

    def forward(self, states, actions_onehot, goals):
        sa = self.encode_sa(states, actions_onehot)
        g = self.encode_g(goals)
        return (sa * g).sum(-1) / self._scale  # diagonal scores


class NoActionCritic(nn.Module):
    """Matched-capacity baseline: phi(state) only (action absent), psi(goal)."""
    def __init__(self, state_dim, goal_dim, emb_dim=EMB_DIM, action_dim=ACTION_DIM):
        super().__init__()
        self.state_dim = state_dim; self.goal_dim = goal_dim
        self.action_dim = action_dim; self.emb_dim = emb_dim
        self.use_action = False
        self.phi = _mlp(state_dim, emb_dim)
        self.psi = _mlp(goal_dim, emb_dim)
        self._scale = emb_dim ** 0.5

    def encode_sa(self, states, actions_onehot=None):
        return self.phi(states)

    def encode_g(self, goals):
        return self.psi(goals)

    def score_matrix(self, sa_repr, g_repr):
        return sa_repr @ g_repr.t() / self._scale

    def forward(self, states, actions_onehot, goals):
        sa = self.phi(states); g = self.psi(goals)
        return (sa * g).sum(-1) / self._scale


def one_hot(actions, n=ACTION_DIM, device=None):
    actions = torch.as_tensor(actions, dtype=torch.long, device=device)
    return torch.nn.functional.one_hot(actions, n).float()


def build_critic(kind, state_dim, goal_dim):
    if kind == "action":
        return StateActionCritic(state_dim, goal_dim)
    if kind == "no_action":
        return NoActionCritic(state_dim, goal_dim)
    raise ValueError(kind)


In [ ]:
# === losses.py ===
"""Eysenbach-style in-batch NCE (sigmoid BCE over the full BxB logit matrix).
NO TD/Bellman/target-net/actor/reward/aux losses.
"""
import torch
import torch.nn.functional as F


def nce_logits(sa_repr, g_repr):
    """logits[i,j] = phi(s_i,a_i) . psi(g_j) / sqrt(D)."""
    d = sa_repr.shape[-1]
    return sa_repr @ g_repr.t() / (d ** 0.5)


def nce_loss(sa_repr, g_repr):
    """Symmetric-free in-batch contrastive: sigmoid BCE, labels = identity matrix.
    Returns (loss, diagnostics dict). Hard-asserts finiteness."""
    B = sa_repr.shape[0]
    logits = nce_logits(sa_repr, g_repr)
    labels = torch.eye(B, device=logits.device)
    loss = F.binary_cross_entropy_with_logits(logits, labels)
    assert torch.isfinite(loss).all(), "non-finite NCE loss"
    with torch.no_grad():
        diag = torch.diagonal(logits)
        off = logits[~torch.eye(B, dtype=torch.bool, device=logits.device)]
        diags = {
            "loss": float(loss.item()),
            "pos_logit_mean": float(diag.mean().item()),
            "neg_logit_mean": float(off.mean().item()),
            "pos_neg_margin": float(diag.mean().item() - off.mean().item()),
            "repr_norm": float(sa_repr.norm(dim=-1).mean().item()),
        }
    return loss, diags


In [ ]:
# === evaluation.py ===
"""Stage-S1 frozen evaluation (PyTorch). Representation metrics, action-shuffle
tests, same-state action sensitivity, and forced-branch alignment. Evaluation only;
never trains. Bootstrap CIs are EPISODE-level.
"""
import numpy as np
import torch


@torch.no_grad()
def _emb(model, states, actions_onehot, goals, device):
    s = torch.as_tensor(states, dtype=torch.float32, device=device)
    g = torch.as_tensor(goals, dtype=torch.float32, device=device)
    if model.use_action:
        a = torch.as_tensor(actions_onehot, dtype=torch.float32, device=device)
        sa = model.encode_sa(s, a)
    else:
        sa = model.encode_sa(s)
    gr = model.encode_g(g)
    return sa, gr


@torch.no_grad()
def diag_scores(model, states, actions_onehot, goals, device="cpu"):
    sa, gr = _emb(model, states, actions_onehot, goals, device)
    return ((sa * gr).sum(-1) / model._scale).cpu().numpy()


@torch.no_grad()
def retrieval_metrics(model, states, actions_onehot, goals, device="cpu", max_n=2048):
    n = min(len(states), max_n)
    sa, gr = _emb(model, states[:n], actions_onehot[:n], goals[:n], device)
    logits = (sa @ gr.t() / model._scale).cpu().numpy()  # [n,n]; col j = goal j
    # for each example i (row), rank of its true goal i among all goals
    ranks = []
    for i in range(n):
        row = logits[i]
        rank = int((row > row[i]).sum()) + 1
        ranks.append(rank)
    ranks = np.array(ranks)
    return {"n": n, "nce_test": _nce_np(logits),
            "top1_acc": float((ranks == 1).mean()), "top5_acc": float((ranks <= 5).mean()),
            "mean_rank": float(ranks.mean()), "mrr": float((1.0 / ranks).mean()),
            "chance_top1": 1.0 / n,
            "pos_logit_mean": float(np.diagonal(logits).mean()),
            "neg_logit_mean": float(logits[~np.eye(n, dtype=bool)].mean())}


def _nce_np(logits):
    n = logits.shape[0]
    lab = np.eye(n)
    z = logits
    # stable BCE-with-logits
    loss = np.maximum(z, 0) - z * lab + np.log1p(np.exp(-np.abs(z)))
    return float(loss.mean())


def _boot_ci(values_per_ep, n_boot=2000, seed=0):
    keys = list(values_per_ep.keys())
    rng = np.random.RandomState(seed)
    stats = []
    for _ in range(n_boot):
        s = rng.choice(keys, size=len(keys), replace=True)
        vals = np.concatenate([values_per_ep[k] for k in s])
        vals = vals[np.isfinite(vals)]
        if len(vals):
            stats.append(vals.mean())
    if not stats:
        return float("nan"), float("nan"), float("nan")
    return float(np.mean(stats)), float(np.percentile(stats, 2.5)), float(np.percentile(stats, 97.5))


@torch.no_grad()
def global_shuffle_delta(model, states, actions, goals, episode_ids, device="cpu", n_boot=2000, seed=0):
    """delta = f(s,a_true,g+) - f(s,a_shuffled,g+), shuffle actions across examples."""
    pass  # inlined: from critic import one_hot
    n = len(states)
    rng = np.random.RandomState(seed)
    perm = rng.permutation(n)
    a_true = one_hot(actions, device=device); a_sh = one_hot(actions[perm], device=device)
    f_true = diag_scores(model, states, a_true.cpu().numpy(), goals, device)
    f_sh = diag_scores(model, states, a_sh.cpu().numpy(), goals, device)
    d = f_true - f_sh
    per_ep = {int(e): d[episode_ids == e] for e in np.unique(episode_ids)}
    m, lo, hi = _boot_ci(per_ep, n_boot, seed)
    return {"delta_global": m, "ci95": [lo, hi], "mean_raw": float(d.mean())}


@torch.no_grad()
def local_shuffle_delta(model, states, actions, goals, episode_ids, support_fn,
                        device="cpu", n_boot=2000, seed=0):
    """Replace true action with a locally-supported ALTERNATIVE (from support_fn)."""
    pass  # inlined: from critic import one_hot
    rng = np.random.RandomState(seed + 1)
    alt = np.array([support_fn(states[i], actions[i], rng) for i in range(len(states))])
    keep = alt >= 0
    a_true = one_hot(actions, device=device).cpu().numpy()
    a_alt = one_hot(np.where(keep, alt, actions), device=device).cpu().numpy()
    f_true = diag_scores(model, states, a_true, goals, device)
    f_alt = diag_scores(model, states, a_alt, goals, device)
    d = (f_true - f_alt)[keep]
    eids = episode_ids[keep]
    per_ep = {int(e): d[eids == e] for e in np.unique(eids)}
    m, lo, hi = _boot_ci(per_ep, n_boot, seed)
    return {"delta_local": m, "ci95": [lo, hi], "n_replaced": int(keep.sum()),
            "mean_raw": float(d.mean()) if keep.any() else None}


@torch.no_grad()
def zero_action_ablation(model, states, actions, goals, device="cpu"):
    pass  # inlined: from critic import one_hot
    a_true = one_hot(actions, device=device).cpu().numpy()
    a_zero = np.zeros_like(a_true)
    f_true = diag_scores(model, states, a_true, goals, device)
    f_zero = diag_scores(model, states, a_zero, goals, device)
    return {"mean_true": float(f_true.mean()), "mean_zero": float(f_zero.mean()),
            "degradation": float(f_true.mean() - f_zero.mean())}


@torch.no_grad()
def same_state_action_sensitivity(model, states, actions, goals, sem_cat, device="cpu", max_n=2048):
    """For each state, score all 18 actions against its true positive goal."""
    pass  # inlined: from critic import one_hot
    n = min(len(states), max_n)
    s = torch.as_tensor(states[:n], dtype=torch.float32, device=device)
    g = torch.as_tensor(goals[:n], dtype=torch.float32, device=device)
    gr = model.encode_g(g)  # [n,D]
    allA = one_hot(np.arange(18), device=device)  # [18,A]
    scores = np.zeros((n, 18))
    for a in range(18):
        aoh = allA[a:a + 1].repeat(n, 1)
        sa = model.encode_sa(s, aoh)
        scores[:, a] = ((sa * gr).sum(-1) / model._scale).cpu().numpy()
    std = scores.std(1); span = scores.max(1) - scores.min(1)
    true_rank = np.array([int((scores[i] > scores[i, actions[i]]).sum()) + 1 for i in range(n)])
    near_flat = float((std < 1e-3).mean())
    return {"n": n, "mean_std_across_actions": float(std.mean()),
            "mean_top_minus_bottom": float(span.mean()),
            "true_action_rank_mean": float(true_rank.mean()),
            "frac_states_near_identical": near_flat,
            "true_action_rank_hist": np.bincount(true_rank, minlength=19)[1:].tolist(),
            "action_score_matrix": scores.tolist()}


@torch.no_grad()
def forced_branch_alignment(model, anchor_states, future_goals, valid, sem_cat,
                            device="cpu", n_boot=2000, seed=0):
    """M[i,j] = f(anchor_state, action_i, future_goal_j) for locally-supported actions.
    future_goals: [n_anchor,18,goal_dim]; valid: [n_anchor,18]."""
    pass  # inlined: from critic import one_hot
    nA = anchor_states.shape[0]
    diag_margins = {}; top1 = []; top2 = []; chance = []; perm_better = []; pair_correct = []
    norm_mats = []; example_mats = []
    rng = np.random.RandomState(seed)
    s_all = torch.as_tensor(anchor_states, dtype=torch.float32, device=device)
    for i in range(nA):
        supp = np.where(valid[i] == 1)[0]
        K = len(supp)
        if K < 2:
            continue
        s = s_all[i:i + 1].repeat(K, 1)
        aoh = one_hot(supp, device=device)
        sa = model.encode_sa(s, aoh)  # [K,D]
        g = torch.as_tensor(future_goals[i, supp], dtype=torch.float32, device=device)
        gr = model.encode_g(g)  # [K,D]
        M = (sa @ gr.t() / model._scale).cpu().numpy()  # [K,K]; row=action, col=goal
        diag = np.diagonal(M)
        off = M[~np.eye(K, dtype=bool)]
        diag_margins[i] = diag - off.mean() if len(off) else diag
        # branch-action matching: for goal j, predicted action = argmax_i M[i,j]
        pred = M.argmax(0)
        top1.append((pred == np.arange(K)).mean())
        top2_hit = np.mean([int(np.argsort(-M[:, j])[:2].tolist().count(j) > 0) for j in range(K)])
        top2.append(top2_hit)
        chance.append(1.0 / K)
        # permutation test: shuffle goal columns, recompute top1
        pcount = 0
        for _ in range(200):
            cols = rng.permutation(K)
            pc = (M[:, cols].argmax(0) == np.arange(K)).mean()
            if pc >= (pred == np.arange(K)).mean():
                pcount += 1
        perm_better.append(pcount / 200.0)
        # pairwise ranking
        cnt = 0; tot = 0
        for a in range(K):
            for b in range(a + 1, K):
                tot += 1
                if M[a, a] > M[b, a] and M[b, b] > M[a, b]:
                    cnt += 1
        pair_correct.append(cnt / max(tot, 1))
        Mn = (M - M.min()) / (M.max() - M.min() + 1e-9)
        norm_mats.append(Mn if Mn.shape == (max(2, K), max(2, K)) else None)
        if len(example_mats) < 6:
            example_mats.append({"anchor": int(i), "supported_actions": supp.tolist(), "M": M.tolist()})
    if not diag_margins:
        return {"insufficient": True}
    m, lo, hi = _boot_ci({k: np.atleast_1d(v) for k, v in diag_margins.items()}, n_boot, seed)
    # aggregate normalized matrix over anchors with the modal K
    Ks = [m_.shape[0] for m_ in norm_mats if m_ is not None]
    aggK = max(set(Ks), key=Ks.count) if Ks else 0
    agg = np.nanmean([m_ for m_ in norm_mats if m_ is not None and m_.shape[0] == aggK], axis=0).tolist() if aggK else None
    return {"n_anchors_eligible": len(diag_margins),
            "diagonal_margin_mean": m, "diagonal_margin_ci95": [lo, hi],
            "top1_matching": float(np.mean(top1)), "top2_matching": float(np.mean(top2)),
            "chance_level": float(np.mean(chance)),
            "perm_test_pvalue": float(np.mean(perm_better)),
            "pairwise_ranking": float(np.mean(pair_correct)),
            "aggregate_matrix_K": aggK, "aggregate_normalized_matrix": agg,
            "example_matrices": example_mats}


In [ ]:
# === validate_colab_pack.py (hash/dim/leakage validator) ===
"""Validate the Stage-S1 Colab pack. Fails on hash mismatch, wrong dims, NaN/Inf,
action out of [0,17], episode leakage across split, invalid H=16 future, or
schema/column mismatch. Used locally AND mirrored in the Colab notebook.
Stdlib + numpy only (no torch / no env deps).
"""
import sys, json, hashlib, zipfile, io
import numpy as np


def load_pack(zip_path):
    z = zipfile.ZipFile(zip_path)
    raw = {n: z.read(n) for n in z.namelist()}
    def arr(name):
        return np.load(io.BytesIO(raw[name]), allow_pickle=False)
    def js(name):
        return json.loads(raw[name].decode())
    return raw, arr, js


def validate(zip_path, strict=True):
    raw, arr, js = load_pack(zip_path)
    errors = []; checks = {}

    manifest = js("manifest.json"); schema = js("feature_schema.json")
    split = js("split_manifest.json"); norm = js("normalization.json")

    # 1) file hashes
    for name, expected in manifest["file_sha256"].items():
        if name == "manifest.json":
            continue
        got = hashlib.sha256(raw[name]).hexdigest()
        if got != expected:
            errors.append(f"hash mismatch: {name}")
    checks["hashes_ok"] = not any("hash mismatch" in e for e in errors)

    S = arr("observational/states.npy"); A = arr("observational/actions.npy")
    GNP = arr("observational/goals_no_player_H16.npy")
    GWO = arr("observational/goals_world_only_H16.npy")
    EID = arr("observational/episode_ids.npy"); TS = arr("observational/timesteps.npy")

    # 2) dims & schema
    if S.shape[1] != schema["state_dim"] or schema["state_dim"] != len(schema["state_schema"]):
        errors.append("state_dim mismatch")
    if GNP.shape[1] != schema["np_dim"] or GWO.shape[1] != schema["wo_dim"]:
        errors.append("goal_dim mismatch")
    if not (len(S) == len(A) == len(GNP) == len(GWO) == len(EID) == len(TS)):
        errors.append("observational length mismatch")
    checks["dims_ok"] = not any(("dim" in e or "length" in e) for e in errors)

    # 3) NaN/Inf
    for nm, a in [("states", S), ("goals_np", GNP), ("goals_wo", GWO)]:
        if not np.isfinite(a).all():
            errors.append(f"non-finite in {nm}")
    checks["finite_ok"] = not any("non-finite" in e for e in errors)

    # 4) action range
    if A.min() < 0 or A.max() > 17:
        errors.append("action outside [0,17]")
    checks["action_range_ok"] = not (A.min() < 0 or A.max() > 17)

    # 5) episode leakage across split
    tr = set(split["train_episode_ids"]); va = set(split["val_episode_ids"]); te = set(split["test_episode_ids"])
    if (tr & va) or (tr & te) or (va & te):
        errors.append("episode leakage across split")
    obs_eps = set(int(x) for x in np.unique(EID))
    if not obs_eps <= (tr | va | te):
        errors.append("observational episode not assigned to a split")
    checks["no_episode_leakage"] = not any("leakage" in e or "not assigned" in e for e in errors)

    # 6) valid H=16 future: goals finite (already) + transitions exist
    checks["has_transitions"] = bool(len(S) > 0)
    if len(S) == 0:
        errors.append("no transitions")

    # 7) branch pack
    AS = arr("branches/anchor_states.npy"); FNP = arr("branches/future_no_player_H16.npy")
    FWO = arr("branches/future_world_only_H16.npy"); VM = arr("branches/valid_mask.npy")
    SC = arr("branches/local_support_counts.npy"); SEM = arr("branches/semantic_action_categories.npy")
    if AS.shape[1] != schema["state_dim"]:
        errors.append("anchor state_dim mismatch")
    if FNP.shape[1:] != (18, schema["np_dim"]) or FWO.shape[1:] != (18, schema["wo_dim"]):
        errors.append("branch future dims mismatch")
    if VM.shape != (AS.shape[0], 18):
        errors.append("valid_mask shape mismatch")
    # valid branch futures must be finite where valid
    vm = VM.astype(bool)
    if vm.any() and not np.isfinite(FNP[vm]).all():
        errors.append("non-finite valid branch np future")
    checks["branch_ok"] = not any("branch" in e or "anchor" in e or "valid_mask" in e for e in errors)
    checks["n_valid_branches"] = int(VM.sum())

    summary = {"zip": zip_path, "n_transitions": int(len(S)), "state_dim": int(S.shape[1]),
               "np_dim": int(GNP.shape[1]), "wo_dim": int(GWO.shape[1]),
               "n_episodes": len(obs_eps), "n_anchors": int(AS.shape[0]),
               "n_valid_branches": int(VM.sum()),
               "split": {"train": len(tr), "val": len(va), "test": len(te)},
               "checks": checks, "errors": errors, "PASS": len(errors) == 0}
    if strict and errors:
        print(json.dumps(summary, indent=2))
        raise AssertionError(f"pack validation FAILED: {errors}")
    return summary


## Section 2 — Load the frozen pack (verify hashes before continuing)


In [ ]:
# Option A: upload (Colab):
#   from google.colab import files; up = files.upload(); PACK = list(up.keys())[0]
# Option B: Google Drive path:
#   PACK = '/content/drive/MyDrive/seaquest_s1_colab_pack.zip'
# Option C: local path (when running outside Colab):
PACK = os.environ.get("S1_PACK", "artifacts/seaquest/stage_s1/seaquest_s1_colab_pack.zip")

summary = validate(PACK, strict=True)   # raises on any failure
print("PACK VALIDATION:", "PASS" if summary["PASS"] else "FAIL")
print(json.dumps({k: summary[k] for k in ["n_transitions","state_dim","np_dim","wo_dim","n_episodes","n_anchors","n_valid_branches","split"]}, indent=2))

raw, arr, js = load_pack(PACK)
MANIFEST = js("manifest.json"); SCHEMA = js("feature_schema.json")
NORM = js("normalization.json"); SPLIT = js("split_manifest.json")
S  = arr("observational/states.npy").astype(np.float32)
A  = arr("observational/actions.npy").astype(np.int64)
GNP = arr("observational/goals_no_player_H16.npy").astype(np.float32)
GWO = arr("observational/goals_world_only_H16.npy").astype(np.float32)
EID = arr("observational/episode_ids.npy").astype(np.int64)
TS  = arr("observational/timesteps.npy").astype(np.int64)
B_AS  = arr("branches/anchor_states.npy").astype(np.float32)
B_FNP = arr("branches/future_no_player_H16.npy").astype(np.float32)
B_FWO = arr("branches/future_world_only_H16.npy").astype(np.float32)
B_VM  = arr("branches/valid_mask.npy").astype(np.int64)
B_SC  = arr("branches/local_support_counts.npy").astype(np.int64)
B_SEM = arr("branches/semantic_action_categories.npy").astype(np.int64)
STATE_DIM = S.shape[1]; NP_DIM = GNP.shape[1]; WO_DIM = GWO.shape[1]
print("source git commit:", MANIFEST["source_git_commit"], "| pack sha (manifest file):",
      hashlib.sha256(raw["manifest.json"]).hexdigest()[:16])


## Section 3 — Inspect data


In [ ]:
print("state_dim", STATE_DIM, "np_dim", NP_DIM, "wo_dim", WO_DIM)
print("transitions", len(S), "anchors", len(B_AS), "valid branches", int(B_VM.sum()))
print("action counts:", np.bincount(A, minlength=18).tolist())
tr=set(SPLIT["train_episode_ids"]); va=set(SPLIT["val_episode_ids"]); te=set(SPLIT["test_episode_ids"])
print("split episodes -> train", len(tr), "val", len(va), "test", len(te))
print("split transitions -> train", int(np.isin(EID,list(tr)).sum()),
      "val", int(np.isin(EID,list(va)).sum()), "test", int(np.isin(EID,list(te)).sum()))
print("state features:", SCHEMA["state_schema"])
print("no_player keys:", SCHEMA["no_player_keys"]); print("world_only keys:", SCHEMA["world_only_keys"])
assert np.isfinite(S).all() and np.isfinite(GNP).all() and np.isfinite(GWO).all(), "non-finite (should be excluded at export)"
print("missing/censoring: dataset pre-filtered at export (no NaN; valid H=16 future; no boundary cross).")


## Section 4 — Pre-training unit tests


In [ ]:
def _norm_states(x): return (x - np.array(NORM["state_mean"])) / np.array(NORM["state_std"])
def _norm_goals(x, view):
    m = NORM[view+"_mean"]; s = NORM[view+"_std"]; return (x - np.array(m)) / np.array(s)

set_seed(0)
m = StateActionCritic(STATE_DIM, NP_DIM).to(DEVICE)
s = torch.randn(4, STATE_DIM, device=DEVICE); g = torch.randn(4, NP_DIM, device=DEVICE)
a1 = one_hot(np.array([0,1,2,3]), device=DEVICE); a2 = one_hot(np.array([5,6,7,8]), device=DEVICE)
o1 = m.encode_sa(s, a1); o2 = m.encode_sa(s, a2)
t1 = (o1 - o2).abs().max().item() > 1e-6
t2 = torch.allclose(m.encode_sa(s, a1), m.encode_sa(s, a1))         # identical input -> identical
t3 = not torch.allclose(torch.cat([s,a1],-1), torch.cat([s,a2],-1)) # permuting action changes input
loss,_ = nce_loss(m.encode_sa(s,a1), m.encode_g(g)); loss.backward()
gw = m.phi[0].weight.grad[:, STATE_DIM:]                            # action-connected columns
t4 = gw.abs().sum().item() > 0
t5 = (B_FNP.shape[1:] == (18, NP_DIM)) and (B_VM.shape == (len(B_AS),18))
t6 = ("train" in NORM.get("fit_on","")) and (set(SPLIT["test_episode_ids"]) and True)  # norm fit on train only
print(dict(action_changes_phi=t1, identical_input_identical=t2, permute_changes_input=t3,
           gradients_reach_action_weights=t4, branch_shapes_valid=bool(t5), norm_excludes_test=bool(t6)))
assert all([t1,t2,t3,t4,t5,t6]), "unit tests failed"
print("UNIT TESTS PASS")


## Section 5 — Train seed 0 (Model A action/no_player, B no-action/no_player, C action/world_only)


In [ ]:
def make_split_tensors(goals_view):
    Sn = _norm_states(S).astype(np.float32)
    Gn = _norm_goals(goals_view, "no_player" if goals_view is GNP else "world_only").astype(np.float32)
    tr=np.isin(EID,SPLIT["train_episode_ids"]); va=np.isin(EID,SPLIT["val_episode_ids"]); te=np.isin(EID,SPLIT["test_episode_ids"])
    pack = lambda mask: (torch.tensor(Sn[mask]), torch.tensor(A[mask]), torch.tensor(Gn[mask]), EID[mask])
    return pack(tr), pack(va), pack(te)

def epoch_pass(model, S_t, A_t, G_t, opt=None):
    model.train(opt is not None)
    n=len(S_t); idx=torch.randperm(n) if opt else torch.arange(n)
    tot=0.0; nb=0; diag_acc={}
    for i in range(0, n, CONFIG["batch_size"]):
        b=idx[i:i+CONFIG["batch_size"]]
        if len(b)<2: continue
        s=S_t[b].to(DEVICE); g=G_t[b].to(DEVICE)
        if model.use_action:
            a=one_hot(A_t[b], device=DEVICE); sa=model.encode_sa(s,a)
        else:
            sa=model.encode_sa(s)
        gr=model.encode_g(g)
        loss,diag=nce_loss(sa,gr)
        if opt:
            opt.zero_grad(); loss.backward()
            gn=torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
            opt.step(); diag["grad_norm"]=float(gn)
        tot+=diag["loss"]; nb+=1
        for k,v in diag.items(): diag_acc[k]=diag_acc.get(k,0)+v
    return {k: v/max(nb,1) for k,v in diag_acc.items()} | {"loss": tot/max(nb,1)}

def train_model(kind, goals_view, seed, tag):
    set_seed(seed)
    (Str,Atr,Gtr,_),(Sva,Ava,Gva,_),_ = make_split_tensors(goals_view)
    gdim = goals_view.shape[1]
    model = build_critic(kind, STATE_DIM, gdim).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    best=1e9; best_state=None; hist=[]; bad=0
    for ep in range(CONFIG["max_epochs"]):
        tr=epoch_pass(model,Str,Atr,Gtr,opt)
        with torch.no_grad(): vl=epoch_pass(model,Sva,Ava,Gva,None)
        hist.append({"epoch":ep,"train_loss":tr["loss"],"val_loss":vl["loss"],
                     "pos_neg_margin":tr.get("pos_neg_margin"),"grad_norm":tr.get("grad_norm")})
        assert math.isfinite(tr["loss"]) and math.isfinite(vl["loss"]), "non-finite loss"
        if vl["loss"]<best-1e-5: best=vl["loss"]; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else: bad+=1
        if bad>=CONFIG["patience"]: break
    model.load_state_dict(best_state)   # validation-selected checkpoint ONLY
    print(f"[{tag}] kind={kind} gdim={gdim} epochs={len(hist)} best_val={best:.4f}")
    return model, hist, best

set_seed(0)
MODELS={}; HIST={}
MODELS["A"],HIST["A"],_ = train_model("action",   GNP, 0, "A action/no_player")
MODELS["B"],HIST["B"],_ = train_model("no_action",GNP, 0, "B no-action/no_player")
MODELS["C"],HIST["C"],_ = train_model("action",   GWO, 0, "C action/world_only")


## Section 6 — Frozen evaluation (validation-selected checkpoints, test set only)


In [ ]:
def test_arrays(goals_view, view_name):
    Sn=_norm_states(S).astype(np.float32); Gn=_norm_goals(goals_view, view_name).astype(np.float32)
    te=np.isin(EID,SPLIT["test_episode_ids"])
    return Sn[te], A[te], Gn[te], EID[te]

# local supported-action sampler from frozen S0.5 branch support (anchor-based NN-free proxy:
# use the per-anchor support; for observational states, fall back to global supported set).
GLOBAL_SUPPORTED = np.where(np.bincount(A, minlength=18) >= max(5, int(0.01*len(A))))[0]
def local_support_fn(state_vec, true_a, rng):
    alt=[a for a in GLOBAL_SUPPORTED if a!=true_a]
    return int(rng.choice(alt)) if alt else -1

EVAL={}
# Model A (no_player)
Ste,Ate,Gte,Ete = test_arrays(GNP,"no_player")
EVAL["A_retrieval"]=retrieval_metrics(MODELS["A"],Ste,one_hot(Ate).numpy(),Gte,DEVICE)
EVAL["A_global_shuffle"]=global_shuffle_delta(MODELS["A"],Ste,Ate,Gte,Ete,DEVICE)
EVAL["A_local_shuffle"]=local_shuffle_delta(MODELS["A"],Ste,Ate,Gte,Ete,local_support_fn,DEVICE)
EVAL["A_zero_action"]=zero_action_ablation(MODELS["A"],Ste,Ate,Gte,DEVICE)
EVAL["A_sensitivity"]=same_state_action_sensitivity(MODELS["A"],Ste,Ate,Gte,B_SEM,DEVICE)
# Model B (no-action baseline) on identical test examples
EVAL["B_retrieval"]=retrieval_metrics(MODELS["B"],Ste,one_hot(Ate).numpy(),Gte,DEVICE)
# Model C (world_only)
Stw,Atw,Gtw,Etw = test_arrays(GWO,"world_only")
EVAL["C_retrieval"]=retrieval_metrics(MODELS["C"],Stw,one_hot(Atw).numpy(),Gtw,DEVICE)
# forced-branch alignment (branch goals normalized with the SAME train stats)
BFNP=( B_FNP - np.array(NORM["no_player_mean"]) ) / np.array(NORM["no_player_std"])
BFWO=( B_FWO - np.array(NORM["world_only_mean"]) ) / np.array(NORM["world_only_std"])
BASn=_norm_states(B_AS).astype(np.float32)
EVAL["A_branch_no_player"]=forced_branch_alignment(MODELS["A"],BASn,BFNP.astype(np.float32),B_VM,B_SEM,DEVICE)
EVAL["C_branch_world_only"]=forced_branch_alignment(MODELS["C"],BASn,BFWO.astype(np.float32),B_VM,B_SEM,DEVICE)
# negative controls on no_player branch alignment
set_seed(123); rand_model=build_critic("action",STATE_DIM,NP_DIM).to(DEVICE)
EVAL["ctrl_random_branch"]=forced_branch_alignment(rand_model,BASn,BFNP.astype(np.float32),B_VM,B_SEM,DEVICE)
EVAL["ctrl_no_action_branch"]=forced_branch_alignment(MODELS["B"],BASn,BFNP.astype(np.float32),B_VM,B_SEM,DEVICE)
print(json.dumps({"A_top1":EVAL["A_retrieval"]["top1_acc"],
                  "A_global_delta":EVAL["A_global_shuffle"]["delta_global"],
                  "A_global_ci":EVAL["A_global_shuffle"]["ci95"],
                  "A_local_delta":EVAL["A_local_shuffle"]["delta_local"],
                  "A_local_ci":EVAL["A_local_shuffle"]["ci95"],
                  "A_zero_degrade":EVAL["A_zero_action"]["degradation"],
                  "A_branch_diag":EVAL["A_branch_no_player"].get("diagonal_margin_mean"),
                  "A_branch_top1":EVAL["A_branch_no_player"].get("top1_matching"),
                  "A_branch_pair":EVAL["A_branch_no_player"].get("pairwise_ranking")}, indent=2))


## Section 7 — Seed-0 gates (C1–C5) and outcome


In [ ]:
def gate_C1():
    h=HIST["A"]; vls=[x["val_loss"] for x in h]
    improved = vls[-1] < vls[0]
    pos_margin = (h[-1]["pos_neg_margin"] or 0) > 0
    above_chance = EVAL["A_retrieval"]["top1_acc"] > EVAL["A_retrieval"]["chance_top1"]
    finite = all(math.isfinite(x["val_loss"]) for x in h)
    return dict(passed=bool(improved and pos_margin and above_chance and finite),
                improved=improved, pos_margin=pos_margin, above_chance=above_chance)
def gate_C2():
    g=EVAL["A_global_shuffle"]; l=EVAL["A_local_shuffle"]; z=EVAL["A_zero_action"]
    return dict(passed=bool(g["ci95"][0]>0 and l["ci95"][0]>0 and z["degradation"]>0),
                global_lb=g["ci95"][0], local_lb=l["ci95"][0], zero_degrade=z["degradation"])
def gate_C3():
    a=EVAL["A_retrieval"]; b=EVAL["B_retrieval"]
    wins = sum([a["nce_test"]<b["nce_test"], a["top1_acc"]>b["top1_acc"],
                (a["pos_logit_mean"]-a["neg_logit_mean"])>(b["pos_logit_mean"]-b["neg_logit_mean"])])
    return dict(passed=bool(wins>=2), wins=wins,
                nce_A=a["nce_test"], nce_B=b["nce_test"], top1_A=a["top1_acc"], top1_B=b["top1_acc"])
def gate_C4():
    e=EVAL["A_branch_no_player"]
    if e.get("insufficient"): return dict(passed=False, reason="insufficient anchors")
    return dict(passed=bool(e["diagonal_margin_ci95"][0]>0 and e["perm_test_pvalue"]<0.05 and e["pairwise_ranking"]>0.5),
                diag_lb=e["diagonal_margin_ci95"][0], top1=e["top1_matching"], chance=e["chance_level"],
                perm_p=e["perm_test_pvalue"], pairwise=e["pairwise_ranking"])
def gate_C5():
    e=EVAL["C_branch_world_only"]
    if e.get("insufficient"): return dict(passed=False)
    crit=[e["diagonal_margin_ci95"][0]>0, e["top1_matching"]>e["chance_level"], e["pairwise_ranking"]>0.5]
    return dict(passed=bool(sum(crit)>=2), diag_lb=e["diagonal_margin_ci95"][0],
                top1=e["top1_matching"], chance=e["chance_level"], pairwise=e["pairwise_ranking"])

GATES={"C1":gate_C1(),"C2":gate_C2(),"C3":gate_C3(),"C4":gate_C4(),"C5":gate_C5()}
def outcome():
    if not GATES["C1"]["passed"]: return "STOP_CRITIC_OPTIMIZATION_FAILURE"
    if not GATES["C2"]["passed"]: return "STOP_CRITIC_IGNORES_ACTION"
    if not GATES["C3"]["passed"]: return "STOP_STATE_EXPLAINS_ALL_SIGNAL"
    if not GATES["C4"]["passed"]: return "STOP_ACTION_SENSITIVITY_NOT_DYNAMICS_ALIGNED"
    if not GATES["C5"]["passed"]: return "PROCEED_ACTION_LEARNED_NO_PLAYER_ONLY"
    return "PROCEED_ACTION_LEARNED_WORLD_ONLY"
OUTCOME=outcome()
print(json.dumps({k:v["passed"] for k,v in GATES.items()}, indent=2)); print("OUTCOME:", OUTCOME)


## Section 8 — Figures, metrics, SUMMARY, and downloadable ZIP


In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
OUT="stage_s1_outputs"; os.makedirs(OUT+"/figures", exist_ok=True); os.makedirs(OUT+"/checkpoints", exist_ok=True)
for tag,mdl in MODELS.items(): torch.save(mdl.state_dict(), f"{OUT}/checkpoints/model_{tag}.pt")
def save(fig,name): fig.savefig(f"{OUT}/figures/{name}",dpi=110,bbox_inches="tight"); plt.close(fig)

for view,tag in [("A","no_player"),("C","world_only")]:
    h=HIST[view]; fig,ax=plt.subplots(figsize=(6,4))
    ax.plot([x["train_loss"] for x in h],label="train"); ax.plot([x["val_loss"] for x in h],label="val")
    ax.set_title(f"training loss {tag} (Model {view})"); ax.legend(); ax.set_xlabel("epoch"); save(fig,f"training_loss_{tag}.png")
r=EVAL["A_retrieval"]; fig,ax=plt.subplots(figsize=(5,4))
ax.bar(["pos","neg"],[r["pos_logit_mean"],r["neg_logit_mean"]]); ax.set_title("positive vs negative logits (A)"); save(fig,"positive_negative_logits.png")
fig,ax=plt.subplots(figsize=(5,4)); ax.bar(["top1","top5","mrr"],[r["top1_acc"],r["top5_acc"],r["mrr"]]); ax.axhline(r["chance_top1"],ls="--",c="k"); ax.set_title("retrieval (A)"); save(fig,"retrieval_metrics.png")
for key,nm in [("A_global_shuffle","global_action_shuffle_delta"),("A_local_shuffle","local_action_shuffle_delta")]:
    e=EVAL[key]; d=e.get("delta_global",e.get("delta_local")); ci=e["ci95"]
    fig,ax=plt.subplots(figsize=(5,4)); ax.bar([nm],[d],yerr=[[d-ci[0]],[ci[1]-d]],capsize=6); ax.axhline(0,c="k"); ax.set_title(nm); save(fig,nm+".png")
sen=EVAL["A_sensitivity"]; sc=np.array(sen["action_score_matrix"])
fig,ax=plt.subplots(figsize=(6,4)); ax.hist(sc.std(1),bins=40); ax.set_title("same-state action score std"); save(fig,"same_state_action_score_spread.png")
fig,ax=plt.subplots(figsize=(6,4)); ax.bar(range(1,19),sen["true_action_rank_hist"]); ax.set_title("true-action rank histogram"); save(fig,"true_action_rank_histogram.png")
fig,ax=plt.subplots(figsize=(5,4)); ax.bar(["A_nce","B_nce"],[EVAL["A_retrieval"]["nce_test"],EVAL["B_retrieval"]["nce_test"]]); ax.set_title("no-action baseline comparison (test NCE)"); save(fig,"no_action_baseline_comparison.png")
for key,nm in [("A_branch_no_player","forced_branch_matrix_no_player"),("C_branch_world_only","forced_branch_matrix_world_only")]:
    e=EVAL[key]; M=e.get("aggregate_normalized_matrix")
    if M is not None:
        fig,ax=plt.subplots(figsize=(5,5)); im=ax.imshow(np.array(M),cmap="viridis"); plt.colorbar(im,ax=ax); ax.set_title(nm+f" (K={e['aggregate_matrix_K']})"); ax.set_xlabel("forced future goal j"); ax.set_ylabel("action i"); save(fig,nm+".png")
fig,ax=plt.subplots(figsize=(7,4))
labels=["diag_margin","top1","pairwise"]; npv=[EVAL["A_branch_no_player"].get(k) for k in ["diagonal_margin_mean","top1_matching","pairwise_ranking"]]
wov=[EVAL["C_branch_world_only"].get(k) for k in ["diagonal_margin_mean","top1_matching","pairwise_ranking"]]
x=np.arange(3); ax.bar(x-0.2,npv,0.4,label="no_player"); ax.bar(x+0.2,wov,0.4,label="world_only"); ax.set_xticks(x); ax.set_xticklabels(labels); ax.legend(); ax.set_title("forced-branch alignment metrics"); save(fig,"forced_branch_alignment_metrics.png")

json.dump({"config":CONFIG,"gates":GATES,"outcome":OUTCOME,"eval":{k:(v if not isinstance(v,dict) else {kk:vv for kk,vv in v.items() if kk not in ("action_score_matrix","example_matrices","aggregate_normalized_matrix")}) for k,v in EVAL.items()},"history":HIST,"manifest_commit":MANIFEST["source_git_commit"]}, open(f"{OUT}/metrics.json","w"), indent=2, default=float)
np.savez_compressed(f"{OUT}/raw_eval_arrays.npz", action_score_matrix=np.array(sen["action_score_matrix"]),
                    A_branch_agg=np.array(EVAL["A_branch_no_player"].get("aggregate_normalized_matrix") or []),
                    C_branch_agg=np.array(EVAL["C_branch_world_only"].get("aggregate_normalized_matrix") or []))

def q(b): return "YES" if b else "NO"
g=GATES
rep=f'''# Seaquest Stage-S1 — Vanilla State Critic — SUMMARY

**OUTCOME: `{OUTCOME}`**  (seed 0; pack commit {MANIFEST["source_git_commit"]})

1. Did the critic optimize normally? {q(g["C1"]["passed"])} (val improved, margin>0, retrieval>chance, finite).
2. Did it learn future-state discrimination? top1={EVAL["A_retrieval"]["top1_acc"]:.3f} (chance {EVAL["A_retrieval"]["chance_top1"]:.4f}), top5={EVAL["A_retrieval"]["top5_acc"]:.3f}, mrr={EVAL["A_retrieval"]["mrr"]:.3f}.
3. Did GLOBAL action shuffle reduce matched scores? delta={EVAL["A_global_shuffle"]["delta_global"]:.4f} CI{EVAL["A_global_shuffle"]["ci95"]}.
4. Did LOCAL supported-action replacement reduce matched scores? delta={EVAL["A_local_shuffle"]["delta_local"]:.4f} CI{EVAL["A_local_shuffle"]["ci95"]}.
5. Did zeroing action degrade performance? degradation={EVAL["A_zero_action"]["degradation"]:.4f}.
6. Did action-conditioned beat no-action baseline? wins={g["C3"]["wins"]}/3 (A nce {g["C3"]["nce_A"]:.4f} vs B {g["C3"]["nce_B"]:.4f}).
7. Same-state action score variation: mean std={EVAL["A_sensitivity"]["mean_std_across_actions"]:.4f}, top-bottom={EVAL["A_sensitivity"]["mean_top_minus_bottom"]:.4f}, near-flat frac={EVAL["A_sensitivity"]["frac_states_near_identical"]:.3f}.
8. Does variation align with forced branches (no_player)? diag_margin={EVAL["A_branch_no_player"].get("diagonal_margin_mean")} CI{EVAL["A_branch_no_player"].get("diagonal_margin_ci95")}, top1={EVAL["A_branch_no_player"].get("top1_matching")} (chance {EVAL["A_branch_no_player"].get("chance_level")}), pairwise={EVAL["A_branch_no_player"].get("pairwise_ranking")}, perm_p={EVAL["A_branch_no_player"].get("perm_test_pvalue")}.
9. Does alignment survive in world_only? diag_margin={EVAL["C_branch_world_only"].get("diagonal_margin_mean")}, top1={EVAL["C_branch_world_only"].get("top1_matching")}, pairwise={EVAL["C_branch_world_only"].get("pairwise_ranking")}.
10. Gates: C1={q(g["C1"]["passed"])} C2={q(g["C2"]["passed"])} C3={q(g["C3"]["passed"])} C4={q(g["C4"]["passed"])} C5={q(g["C5"]["passed"])}.
11. Is the action-learning problem solved for this state critic? {"YES (no_player; world_only too)" if OUTCOME=="PROCEED_ACTION_LEARNED_WORLD_ONLY" else ("PARTIAL (no_player only)" if OUTCOME=="PROCEED_ACTION_LEARNED_NO_PLAYER_ONLY" else "NO — "+OUTCOME)}.

Negative controls (no_player branch diag margin): trained-action={EVAL["A_branch_no_player"].get("diagonal_margin_mean")}, random-init={EVAL["ctrl_random_branch"].get("diagonal_margin_mean")}, no-action={EVAL["ctrl_no_action_branch"].get("diagonal_margin_mean")}.
'''
open(f"{OUT}/SUMMARY.md","w").write(rep); print(rep)
import shutil; shutil.make_archive("seaquest_stage_s1_results","zip",OUT)
print("wrote seaquest_stage_s1_results.zip")
# from google.colab import files; files.download("seaquest_stage_s1_results.zip")


## Section 7b — Seeds [1, 2] (MANUAL; run only if seed-0 passed C1–C4)


In [ ]:
# Guarded: do NOT auto-run after a failure. Confirm seed-0 outcome first.
assert OUTCOME.startswith("PROCEED"), f"seed-0 outcome is {OUTCOME}; do not run more seeds"
EXTRA={}
for sd in [1,2]:
    mA,_,_=train_model("action",GNP,sd,f"A seed{sd}")
    St,At,Gt,Et=test_arrays(GNP,"no_player")
    EXTRA[sd]={"global":global_shuffle_delta(mA,St,At,Gt,Et,DEVICE),
               "branch":forced_branch_alignment(mA,_norm_states(B_AS).astype('float32'),
                        ((B_FNP-np.array(NORM["no_player_mean"]))/np.array(NORM["no_player_std"])).astype('float32'),B_VM,B_SEM,DEVICE)}
print(json.dumps({sd:{"global_ci":EXTRA[sd]["global"]["ci95"],"branch_diag":EXTRA[sd]["branch"].get("diagonal_margin_mean")} for sd in EXTRA}, indent=2))
